(param-model-nb)=
# Parameter-dependent `TBModel`

In this example, we illustrate how to set hoppings and onsite terms symbolically to create a parameter-dependent tight-binding model. 

In [ ]:
from pythtb import TBModel, Lattice
import numpy as np

Let's first make a finite model with two orbitals on a 1D chain. We will set:

- A parameterized onsite energy `"mA"` on orbital A
- A parameterized onsite energy `"mB"` on orbital B
- A parameterized hopping `"t"` between orbitals A and B

In [ ]:
lat_vecs = [[1]]
orb_vecs = [[0], [1 / 2]]
lat = Lattice(lat_vecs=lat_vecs, orb_vecs=orb_vecs)
model = TBModel(lattice=lat, spinful=False)

model.set_onsite("mA", ind_i=0)
model.set_onsite("mB", ind_i=1)
model.set_hop("t", 0, 1)
print(model)

We can utilize the parameterization to generate the Hamiltonian (and any other observables) at different parameter values without needing to redefine the model each time. To do this, we simply pass key word arguments to the relevant methods. We can either pass a single value or an array of values to evaluate the observable at multiple parameter points simultaneously.

In [ ]:
model.parameters

In [ ]:
H = model.hamiltonian(
    mA=0.1, mB=-0.1, t=np.linspace(0, 1, 10)
)  # Hamiltonian with t varying from 0 to 1, mA=0.1, mB=-0.1

# Should be (10, 2, 2) since there are 2 orbitals and 10 t values
print(f"H shape: {H.shape}")

Passing values to observables will not resolve the parameters in the model itself; it only affects the output of the observable being called. The model retains its symbolic parameters until they are explicitly set to numerical values using the `set_parameters` method.

In [ ]:
print(f"Model parameters before setting: {model.parameters}")
model.set_parameters(mA=0.1, mB=-0.1, t=0.5)
print(f"Model parameters after setting: {model.parameters}")

print(model)

## Functionally dependent parameters

We can also define parameters that are callables. These will take a parameter, and return the hopping or onsite value. This allows us to define parameters that depend on other parameters. A convenient way of doing this is using Python's `lambda` functions. 

In [ ]:
model = TBModel(lattice=lat, spinful=False)

model.set_onsite(lambda mA: np.cos(mA), ind_i=0)
model.set_onsite(lambda mB: np.sin(mB), ind_i=1)
model.set_hop(lambda mA, mB: (mA + mB) / 2, 0, 1)
print(model)

In [ ]:
model.parameters

We then resolve the parameters in the same way as before.

In [ ]:
H = model.hamiltonian(
    mA=np.linspace(0, np.pi, 10), mB=np.linspace(0, np.pi, 12)
)  # Hamiltonian with mA and mB varying

# Should be (10, 12, 2, 2) since there are 2 orbitals, 10 mA values and 12 mB values
print(f"H shape: {H.shape}")

In [ ]:
model.set_parameters(mA=np.pi / 4, mB=np.pi / 4)
print(model)